# Simple: Full Masking with PAMOLA.CORE

**Goal**: Learn full masking basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply full field masking using execute()
- Compare before/after results
- Understand complete data obfuscation

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/masking/
        └── 01_full_masking_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.masking.full_masking_op import (
        FullMaskingOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before masking

**What you'll see:**
- Dataset summary (20 records, 8 columns)
- First 5 records with sensitive fields visible
- Column statistics (types, unique counts)
- Sensitive fields: email, phone, ssn, credit_card

**Note:** Sample contains realistic sensitive data patterns suitable for demonstrating complete field masking

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with sensitive fields
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'name': [
            'John Smith', 'Jane Doe', 'Bob Johnson', 'Alice Williams',
            'Charlie Brown', 'Diana Prince', 'Eve Anderson', 'Frank Miller',
            'Grace Lee', 'Henry Wilson', 'Iris Martinez', 'Jack Taylor',
            'Kate Davis', 'Leo Garcia', 'Mary Rodriguez', 'Nick Lopez',
            'Olivia Hernandez', 'Paul Walker', 'Quinn Hall', 'Rose Allen'
        ],
        'email': [
            'john.smith@example.com', 'jane.doe@example.com', 'bob.j@example.com',
            'alice.w@example.com', 'charlie.b@example.com', 'diana.p@example.com',
            'eve.a@example.com', 'frank.m@example.com', 'grace.l@example.com',
            'henry.w@example.com', 'iris.m@example.com', 'jack.t@example.com',
            'kate.d@example.com', 'leo.g@example.com', 'mary.r@example.com',
            'nick.l@example.com', 'olivia.h@example.com', 'paul.w@example.com',
            'quinn.h@example.com', 'rose.a@example.com'
        ],
        'phone': [
            '555-0101', '555-0102', '555-0103', '555-0104', '555-0105',
            '555-0106', '555-0107', '555-0108', '555-0109', '555-0110',
            '555-0111', '555-0112', '555-0113', '555-0114', '555-0115',
            '555-0116', '555-0117', '555-0118', '555-0119', '555-0120'
        ],
        'ssn': [
            '123-45-6789', '234-56-7890', '345-67-8901', '456-78-9012',
            '567-89-0123', '678-90-1234', '789-01-2345', '890-12-3456',
            '901-23-4567', '012-34-5678', '123-45-6780', '234-56-7891',
            '345-67-8902', '456-78-9013', '567-89-0124', '678-90-1235',
            '789-01-2346', '890-12-3457', '901-23-4568', '012-34-5679'
        ],
        'credit_card': [
            '4532-1234-5678-9010', '5425-2345-6789-0123', '3782-3456-7890-1234',
            '6011-4567-8901-2345', '4532-5678-9012-3456', '5425-6789-0123-4567',
            '3782-7890-1234-5678', '6011-8901-2345-6789', '4532-9012-3456-7890',
            '5425-0123-4567-8901', '3782-1234-5678-9012', '6011-2345-6789-0123',
            '4532-3456-7890-1234', '5425-4567-8901-2345', '3782-5678-9012-3456',
            '6011-6789-0123-4567', '4532-7890-1234-5678', '5425-8901-2345-6789',
            '3782-9012-3456-7890', '6011-0123-4567-8901'
        ],
        'department': [
            'Engineering', 'Sales', 'HR', 'Marketing', 'Finance',
            'Operations', 'Engineering', 'Sales', 'HR', 'Marketing',
            'Finance', 'Operations', 'Engineering', 'Sales', 'HR',
            'Marketing', 'Finance', 'Operations', 'Engineering', 'Sales'
        ],
        'salary': [
            75000, 82000, 68000, 91000, 79000, 85000, 72000, 88000, 70000, 95000,
            78000, 83000, 76000, 89000, 71000, 92000, 80000, 86000, 74000, 90000
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Default: `"email"`
   - Change to YOUR sensitive field (email, phone, ssn, credit_card)
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (unique values, sample values, data type, avg length)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Field must exist in dataset. All types (string, numeric, date) supported for masking

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Field to mask - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to mask: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:3])}")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Average length: {df[field_name].astype(str).str.len().mean():.1f} characters")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_masking_001",
    task_type="full_masking",
    description=f"Full masking of '{field_name}' field.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Masking {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration (production defaults set)
- Run to execute full masking
- Monitor progress bar (6 steps: validate → mask → metrics → viz → save → complete)

**Key parameters:**
- `mask_char='*'`: Character used for masking
- `preserve_length=True`: Masked value has same length as original
- `random_mask=False`: Use fixed mask_char (not random)
- `preserve_format=False`: Simple full masking
- `mode='REPLACE'`: Overwrites original field

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: validation → masking → metrics → viz → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (4-6 files expected)

**Note:** Execution takes ~2-5 seconds for 20 records. All original values completely replaced

In [ ]:
# Create operation with production-style configuration
operation = FullMaskingOperation(
    # Core parameters
    field_name=field_name,               # From config
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Masking parameters
    mask_char='*',                       # Character for masking
    preserve_length=True,                # Keep original length
    fixed_length=None,                   # Use original length
    random_mask=False,                   # Use fixed mask character
    mask_char_pool=None,                 # Not used with random_mask=False
    
    # Format handling
    preserve_format=False,               # Simple masking without format preservation
    format_patterns=None,                # Not used when preserve_format=False
    
    # Type-specific handling
    numeric_output='string',             # Output as string (not numeric)
    date_format=None,                    # Not applicable for this field
    
    # Privacy settings
    ka_risk_field=None,
    risk_threshold=5,
    vulnerable_record_strategy='suppress',
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Mask character:   '{operation.mask_char}'")
print(f"  Preserve length:  {operation.preserve_length}")
print(f"  Preserve format:  {operation.preserve_format}")
print(f"  Random mask:      {operation.random_mask}")
print(f"  Numeric output:   {operation.numeric_output}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing full masking operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare masked data
- Review masking effectiveness metrics
- Verify complete data obfuscation

**What you'll see:**
- First 10 masked records (before → after comparison)
- Masking summary: uniqueness reduction %, length preservation
- Masking pattern analysis (mask char used, example values)
- Privacy metrics from operation results

**Validate:**
- ✅ No original values visible in masked output
- ✅ Length preserved (if configured)
- ✅ All records successfully masked
- ✅ Artifacts generated without errors

**Note:** All sensitive data should be completely masked. If original values appear, check configuration

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records side by side
    print("\n🔍 Sample Masked Records (Before → After):")
    print("-" * 80)
    # Show sample records
    print("\n🔍 Sample Generalized Records:")
    display_cols = [field_name]
    if 'record_id' in result_df.columns:
        display_cols.insert(0, 'record_id')
    if 'ssn' in result_df.columns:
        display_cols.append('ssn')
    print(result_df[display_cols].head(10))
    
    # Masking statistics
    original_unique = df[field_name].nunique()
    masked_unique = result_df[field_name].nunique()
    avg_orig_len = df[field_name].astype(str).str.len().mean()
    avg_masked_len = result_df[field_name].astype(str).str.len().mean()
    
    print("\n" + "=" * 80)
    print("✨ MASKING SUMMARY")
    print("=" * 80)
    print(f"  Original unique values: {original_unique}")
    print(f"  Masked unique values:   {masked_unique}")
    print(f"  Uniqueness reduction:   {((original_unique - masked_unique) / original_unique * 100):.1f}%")
    print(f"\n  Original avg length:    {avg_orig_len:.1f} characters")
    print(f"  Masked avg length:      {avg_masked_len:.1f} characters")
    print(f"  Length preserved:       {'Yes' if abs(avg_orig_len - avg_masked_len) < 0.1 else 'No'}")
    
    # Show pattern analysis
    print(f"\n📋 Masking Pattern Analysis:")
    masked_sample = result_df[field_name].iloc[0]
    print(f"  Mask character used:    '{operation.mask_char}'")
    print(f"  Example masked value:   {masked_sample}")
    print(f"  Format preserved:       {operation.preserve_format}")
    
    # Display result metrics
    if result.metrics:
        print("\n🔒 Privacy Metrics:")
        for key, value in list(result.metrics.items())[:10]:  # Show first 10
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 All sensitive data completely masked!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Masked data CSV
├── metrics/          # Masking effectiveness metrics
├── visualizations/   # Before/after comparison charts
└── config/           # Operation configuration
```

**Note:** Output CSV contains fully masked sensitive field. Ready for production use or sharing

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Effectiveness ratio, masking rate, performance stats
2. **Output Data**: Side-by-side comparison (first 15 records) with masking analysis
3. **Visualizations**: Before/after charts displayed inline

**Validate:**
- ✅ All values fully masked (no original data visible)
- ✅ Masking rate = 100%
- ✅ Length preserved (if configured)
- ✅ Only mask characters appear in output

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types inside IF
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display effectiveness metrics
            if 'effectiveness' in metrics:
                print("\n📈 EFFECTIVENESS:")
                eff = metrics['effectiveness']
                print(f"   Total Records: {eff.get('total_records', 'N/A'):,}")
                print(f"   Original Unique: {eff.get('original_unique', 'N/A')}")
                print(f"   Anonymized Unique: {eff.get('anonymized_unique', 'N/A')}")
                print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
            
            # Display performance metrics
            if 'performance' in metrics:
                print("\n⚡ PERFORMANCE:")
                perf = metrics['performance']
                print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {perf.get('records_processed', 0):,}")
                print(f"   Records/Second: {perf.get('records_per_second', 0):.2f}")
            
            # Display masking-specific metrics
            if 'values_masked' in metrics:
                print("\n🎭 MASKING STATISTICS:")
                print(f"   Values Masked: {metrics.get('values_masked', 0):,}")
                print(f"   Masking Rate: {metrics.get('masking_rate', 0):.2%}")
                print(f"   Format Preserved: {metrics.get('format_preserved_count', 0)}")
                print(f"   Conditional Masked: {metrics.get('conditional_mask_count', 0)}")
                print(f"   Risk-Based Masked: {metrics.get('risk_based_mask_count', 0)}")
            
            # Display suppression rate
            if 'suppression_rate' in metrics:
                print(f"\n🔒 PRIVACY:")
                print(f"   Suppression Rate: {metrics.get('suppression_rate', 0):.2%}")
            
            # Display field info
            if 'field_info' in metrics:
                print("\n📊 FIELD COMPARISON:")
                field_info = metrics['field_info']
                
                if 'original' in field_info:
                    orig = field_info['original']
                    print(f"\n   ORIGINAL:")
                    print(f"      Unique Count: {orig.get('unique_count', 0)}")
                    print(f"      Null Count: {orig.get('null_count', 0)}")
                
                if 'processed' in field_info:
                    proc = field_info['processed']
                    print(f"\n   MASKED:")
                    print(f"      Unique Count: {proc.get('unique_count', 0)}")
                    print(f"      Null Count: {proc.get('null_count', 0)}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Sample (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (Side-by-Side Comparison):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            
            # Create comparison view
            if field_name in output_df.columns and field_name in df.columns:
                print("📊 Before/After Comparison (First 15 records):")
                print("-" * 80)

                has_record_id = 'record_id' in df.columns and 'record_id' in output_df.columns

                # Case 1: record_id exists → try to match IDs
                if has_record_id:
                    ids = df['record_id'].head(15)
                    valid_ids = ids[ids.isin(output_df['record_id'])]

                    df_valid = df[df['record_id'].isin(valid_ids)]
                    out_valid = output_df[output_df['record_id'].isin(valid_ids)]

                    # If no matching IDs → still build comparison but without ID column
                    if valid_ids.empty:
                        comparison = pd.DataFrame({
                            'Original': df[field_name].head(15).values,
                            'Masked': output_df[field_name].head(15).values,
                            'Length': output_df[field_name].head(15).astype(str).str.len().values
                        })
                    else:
                        comparison = pd.DataFrame({
                            'ID': df_valid['record_id'].values,
                            'Original': df_valid[field_name].values,
                            'Masked': out_valid[field_name].values,
                            'Length': out_valid[field_name].astype(str).str.len().values
                        })

                # Case 2: record_id does NOT exist → do comparison without ID column
                else:
                    comparison = pd.DataFrame({
                        'Original': df[field_name].head(15).values,
                        'Masked': output_df[field_name].head(15).values,
                        'Length': output_df[field_name].head(15).astype(str).str.len().values
                    })

                print(comparison.to_string(index=False))

                # Analysis
                print(f"\n📈 MASKING ANALYSIS:")
                masked_chars = set(''.join(output_df[field_name].astype(str).unique()))
                print(f"   Unique masked characters: {sorted(masked_chars)}")
                print(f"   All values fully masked: {len(set(df[field_name]) & set(output_df[field_name])) == 0}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Visualizations (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV with sensitive fields  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review masked data and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end masking
- Field name is configurable via `create_config_kwargs()`
- Full masking replaces all characters with mask_char
- Length can be preserved for format consistency
- Original data is completely obfuscated
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_full_masking_advanced.ipynb`** includes:
- Format preservation with regex patterns
- Random masking with character pools
- Fixed-length masking
- Numeric output options
- Date format masking
- Conditional masking
- Risk-based masking with k-anonymity
- Performance optimization

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Masking Patterns Guide](../../docs/masking_patterns.md)